# AlphaCluster — Optuna Tuning for Simple Actions

Tunes reward and PPO hyperparameters for the **3-action model** (long/flat/short)
using the **standard 19-feature observation space (5 OHLCV + 14 technical indicators)**.

## Key goals

1. **Fix short bias** — model picks short 99.7% of the time. Higher entropy + direction balance scoring.
2. **Find optimal reward scales** for the 3-action space (simpler than 36-action).
3. **Tune ent_coef aggressively** — wider range, critical for exploration symmetry.

## Design

- **12 parameters**: 8 reward scales + lr + gamma + ent_coef_phase1 + phase1_end
- **Phase 1 (screening)**: 200k steps × 25 trials (TPE, seeded with v1 best + variants)
- **Phase 2 (validation)**: 500k steps × top 10
- **Scoring**: composite of PnL, trade activity, win rate, **direction balance**
- Persists to Google Drive with resume support

## Cell 1 — Mount Drive & Install Dependencies

In [ ]:
import os
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/alphacluster")
DRIVE_SRC = DRIVE_ROOT / "src"

assert DRIVE_SRC.exists(), (
    f"Source not found at {DRIVE_SRC}\n"
    f"Copy your local src/ directory to Google Drive: My Drive/alphacluster/src/"
)

LOCAL_SRC = Path("/content/src")
if LOCAL_SRC.exists():
    shutil.rmtree(LOCAL_SRC)
shutil.copytree(DRIVE_SRC, LOCAL_SRC)
print(f"Copied source to {LOCAL_SRC}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "stable-baselines3>=2.4,<3.0", "gymnasium>=1.0,<2.0",
    "pyarrow", "python-dotenv", "tqdm", "rich",
    "optuna>=3.0", "plotly", "kaleido==0.2.1"], check=True)

sys.path.insert(0, str(LOCAL_SRC))

import alphacluster
print(f"AlphaCluster loaded from {Path(alphacluster.__file__).parent}")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected.")

DRIVE_DIR = Path("/content/drive/MyDrive/AlphaCluster/optuna_simple_v3/")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
STUDY_DB = DRIVE_DIR / "optuna_study.db"
RESULTS_CSV = DRIVE_DIR / "trial_results.csv"
BEST_PARAMS_JSON = DRIVE_DIR / "best_params.json"
PLOTS_DIR = DRIVE_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nPersistence: {DRIVE_DIR}")

## Cell 2 — Load & Split Data

In [ ]:
import pandas as pd

DATA_DIR = Path("/content/drive/MyDrive/alphacluster/data")
KLINES_PATH = DATA_DIR / "btcusdt_5m.parquet"

assert KLINES_PATH.exists(), f"Kline data not found at {KLINES_PATH}"

klines_df = pd.read_parquet(KLINES_PATH, engine="pyarrow")
print(f"Loaded {len(klines_df):,} candles")

n = len(klines_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = klines_df.iloc[:train_end].reset_index(drop=True)
val_df = klines_df.iloc[train_end:val_end].reset_index(drop=True)

print(f"Data split: train={len(train_df):,}  val={len(val_df):,}")

## Cell 3 — Define Objective Function

**Key differences from optuna_v2:**
- Uses `simple_actions=True` (3-action space)
- Scoring includes **direction balance** penalty
- `ent_coef_phase1` range widened to `[0.02, 0.15]`
- `phase1_end` is tunable (longer Phase 1 = more exploration time)
- 12 parameters total

In [ ]:
import logging
import shutil
from collections import Counter

import numpy as np
import optuna

from alphacluster.agent.config import TrainingConfig
from alphacluster.agent.trainer import (
    CurriculumCallback,
    TrainingMetricsCallback,
    create_agent,
)
from alphacluster.env.trading_env import TradingEnv
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize

logger = logging.getLogger("optuna_simple_v3")

SCREENING_TIMESTEPS = 200_000
VALIDATION_TIMESTEPS = 500_000
METRICS_LOG_FREQ = 50_000
N_ENVS = 4
EVAL_EPISODES = 5


class OptunaPruningCallback(BaseCallback):
    """Report intermediate scores to Optuna and prune bad trials."""

    def __init__(self, trial, eval_env, model_ref, report_freq):
        super().__init__(verbose=0)
        self.trial = trial
        self.eval_env = eval_env
        self.model_ref = model_ref
        self.report_freq = report_freq
        self._next_report = report_freq

    def _on_step(self):
        if self.num_timesteps < self._next_report:
            return True
        self._next_report += self.report_freq
        score, _ = evaluate_agent(self.model, self.eval_env, n_episodes=3)
        self.trial.report(score, self.num_timesteps)
        if self.trial.should_prune():
            raise optuna.TrialPruned()
        return True


def make_env(df, config, rank=0):
    """Factory for SubprocVecEnv."""
    def _init():
        import os, sys, warnings
        os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
        warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")

        env = TradingEnv(
            df=df,
            window_size=config.window_size,
            episode_length=config.episode_length,
            simple_actions=True,
            fixed_size_pct=0.10,
            fixed_leverage=10,
        )
        env.reset(seed=rank)
        return env
    return _init


def evaluate_agent(model, eval_env, n_episodes=5):
    """Run deterministic evaluation and return (score, details).

    Score accounts for PnL, trade activity, win rate, and direction balance.
    """
    action_counts = Counter()
    total_pnl = 0.0
    total_trades = 0
    total_wins = 0
    total_steps = 0
    flat_steps = 0
    trade_durations = []
    long_trades = 0
    short_trades = 0

    for ep in range(n_episodes):
        obs, info = eval_env.reset(seed=ep * 42)
        done = False
        prev_n = len(eval_env.account.trade_history)
        open_trade = None
        ep_step = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            action_counts[int(action)] += 1
            obs, reward, terminated, truncated, info = eval_env.step(action)
            done = terminated or truncated
            ep_step += 1
            total_steps += 1

            if eval_env.account.position_side == "flat":
                flat_steps += 1

            th = eval_env.account.trade_history
            while prev_n < len(th):
                entry = th[prev_n]
                prev_n += 1
                if entry["action"] == "open":
                    open_trade = {"side": entry.get("side"), "step": ep_step}
                elif entry["action"] == "close" and open_trade is not None:
                    pnl = entry.get("pnl", 0.0)
                    total_pnl += pnl
                    total_trades += 1
                    if pnl > 0:
                        total_wins += 1
                    duration = ep_step - open_trade["step"]
                    trade_durations.append(duration)
                    if open_trade["side"] == "long":
                        long_trades += 1
                    else:
                        short_trades += 1
                    open_trade = None

    trades_per_ep = total_trades / max(n_episodes, 1)
    win_rate = (total_wins / total_trades * 100) if total_trades > 0 else 0
    flat_pct = (flat_steps / total_steps * 100) if total_steps > 0 else 100
    avg_duration = np.mean(trade_durations) if trade_durations else 0
    pnl_pct = total_pnl / eval_env.initial_balance * 100

    # Direction balance: 1.0 = perfect 50/50, 0.0 = all one direction
    if long_trades + short_trades > 0:
        minority = min(long_trades, short_trades)
        direction_balance = minority / (long_trades + short_trades) * 2  # 0..1
    else:
        direction_balance = 0.0

    # Action entropy (0=always same action, 1=uniform)
    total_actions = sum(action_counts.values())
    if total_actions > 0:
        probs = np.array([action_counts.get(a, 0) / total_actions for a in range(3)])
        probs = probs[probs > 0]
        action_entropy = -np.sum(probs * np.log(probs)) / np.log(3)  # normalized 0..1
    else:
        action_entropy = 0.0

    # ── Rejection filters ──
    if flat_pct > 70:
        return -1000.0, {"reject": "flat_pct > 70%"}
    if trades_per_ep < 2:
        return -1000.0, {"reject": f"trades_per_ep={trades_per_ep:.1f} < 2"}

    # ── Composite score ──
    # PnL component (clipped, centered)
    pnl_norm = np.clip(pnl_pct, -50, 50) / 100 + 0.5  # 0..1

    # Trade activity (prefer 3-30 trades/ep for simple actions)
    trades_norm = min(trades_per_ep, 30) / 30  # 0..1

    # Win rate
    wr_norm = win_rate / 100  # 0..1

    # Direction balance bonus — critical for fixing short bias
    # 0.0 if all one direction, 1.0 if 50/50
    dir_norm = direction_balance  # 0..1

    score = (
        pnl_norm * 0.30
        + trades_norm * 0.15
        + wr_norm * 0.25
        + dir_norm * 0.30  # heavy weight on direction balance
    )

    details = {
        "pnl_pct": round(pnl_pct, 2),
        "trades_per_ep": round(trades_per_ep, 1),
        "win_rate": round(win_rate, 1),
        "flat_pct": round(flat_pct, 1),
        "avg_duration": round(avg_duration, 1),
        "long_trades": long_trades,
        "short_trades": short_trades,
        "direction_balance": round(direction_balance, 3),
        "action_entropy": round(action_entropy, 3),
        "action_dist": {a: action_counts.get(a, 0) for a in range(3)},
    }
    return float(score), details


def objective(trial):
    """Optuna objective: 12 params for simple-action model."""
    train_env = None
    try:
        # ── Reward scales ──
        fee_scale = trial.suggest_float("fee_scale", 0.1, 2.0)
        opp_cost_scale = trial.suggest_float("opportunity_cost_scale", 0.1, 2.0)
        opp_cost_cap = trial.suggest_float("opportunity_cost_cap", 0.01, 0.15)
        opp_cost_threshold = trial.suggest_float(
            "opportunity_cost_threshold", 0.001, 0.005,
        )
        churn_scale = trial.suggest_float("churn_penalty_scale", 0.1, 2.0)
        dd_scale = trial.suggest_float("drawdown_penalty_scale", 0.1, 2.0)
        quality_scale = trial.suggest_float("quality_scale", 0.1, 2.0)
        pos_mgmt_scale = trial.suggest_float("position_mgmt_scale", 0.1, 2.0)

        # ── PPO hyperparameters ──
        lr = trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True)
        gamma = trial.suggest_float("gamma", 0.99, 0.999)

        # ── Entropy & curriculum — key for fixing direction bias ──
        ent = trial.suggest_float("ent_coef_phase1", 0.02, 0.15, log=True)
        phase1_end = trial.suggest_float("phase1_end", 0.2, 0.5)

        base_reward_config = {
            "fee_scale": fee_scale,
            "opportunity_cost_scale": opp_cost_scale,
            "opportunity_cost_cap": opp_cost_cap,
            "opportunity_cost_threshold": opp_cost_threshold,
            "churn_penalty_scale": churn_scale,
            "drawdown_penalty_scale": dd_scale,
            "quality_scale": quality_scale,
            "position_mgmt_scale": pos_mgmt_scale,
        }

        config = TrainingConfig(
            total_timesteps=SCREENING_TIMESTEPS,
            learning_rate=lr,
            gamma=gamma,
            ent_coef=ent,
            simple_actions=True,
            fixed_size_pct=0.10,
            fixed_leverage=10,
            phase1_end=phase1_end,
            phase2_end=phase1_end + 0.3,  # Phase 2 always 30%
            eval_freq=SCREENING_TIMESTEPS + 1,
        )

        envs = SubprocVecEnv([
            make_env(train_df, config, rank=i)
            for i in range(N_ENVS)
        ])
        train_env = VecNormalize(envs, norm_obs=False, norm_reward=True, clip_reward=10.0)

        eval_raw = TradingEnv(
            df=val_df,
            window_size=config.window_size,
            episode_length=config.episode_length,
            simple_actions=True,
            fixed_size_pct=0.10,
            fixed_leverage=10,
        )

        agent = create_agent(train_env, config, verbose=0)

        curriculum_cb = CurriculumCallback(
            config,
            base_reward_config=base_reward_config,
            phase3_fee_multiplier=1.0,
            phase3_churn_multiplier=1.0,
            ent_coef_phase1=ent,
            verbose=0,
        )
        pruning_cb = OptunaPruningCallback(
            trial, eval_raw, agent, report_freq=METRICS_LOG_FREQ,
        )

        agent.learn(
            total_timesteps=config.total_timesteps,
            callback=CallbackList([curriculum_cb, pruning_cb]),
            progress_bar=False,
        )

        score, details = evaluate_agent(agent, eval_raw, n_episodes=EVAL_EPISODES)
        for k, v in details.items():
            if isinstance(v, (int, float, str)):
                trial.set_user_attr(k, v)

        return score

    except optuna.TrialPruned:
        raise
    except Exception as e:
        trial.set_user_attr("error", str(e))
        logger.exception("Trial %d failed: %s", trial.number, e)
        return -1000.0
    finally:
        if train_env is not None:
            try:
                train_env.close()
            except Exception:
                pass


print("Objective function defined (12 params, simple_actions=True, 19 features).")
print("Scoring weights: PnL=0.30, trades=0.15, win_rate=0.25, direction_balance=0.30")

## Cell 4 — Run Screening (200k × 25 trials)

In [ ]:
N_SCREENING_TRIALS = 25

print("=" * 60)
print(f"PHASE 1: Screening — 200k steps x {N_SCREENING_TRIALS} trials")
print("Simple actions (3-action), 19 features, direction balance scoring")
print("=" * 60)

study = optuna.create_study(
    study_name="alphacluster_simple_v3_tuning",
    direction="maximize",
    storage=f"sqlite:///{STUDY_DB}",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=100_000,
    ),
    sampler=optuna.samplers.TPESampler(seed=42),
)

# Seed with high-entropy configs to encourage direction exploration
if len(study.trials) == 0:
    SEED_TRIALS = [
        {  # Baseline: v1 best adapted for simple actions, high entropy
            "fee_scale": 0.10, "opportunity_cost_scale": 1.4,
            "opportunity_cost_cap": 0.03, "opportunity_cost_threshold": 0.002,
            "churn_penalty_scale": 0.9, "drawdown_penalty_scale": 0.9,
            "quality_scale": 1.0, "position_mgmt_scale": 1.2,
            "learning_rate": 1.2e-4, "ent_coef_phase1": 0.10, "gamma": 0.996,
            "phase1_end": 0.4,
        },
        {  # Very high entropy, long Phase 1
            "fee_scale": 0.5, "opportunity_cost_scale": 1.0,
            "opportunity_cost_cap": 0.05, "opportunity_cost_threshold": 0.002,
            "churn_penalty_scale": 0.5, "drawdown_penalty_scale": 0.5,
            "quality_scale": 1.0, "position_mgmt_scale": 1.0,
            "learning_rate": 3e-4, "ent_coef_phase1": 0.15, "gamma": 0.995,
            "phase1_end": 0.5,
        },
        {  # Low opp cost, moderate entropy
            "fee_scale": 0.3, "opportunity_cost_scale": 0.3,
            "opportunity_cost_cap": 0.02, "opportunity_cost_threshold": 0.003,
            "churn_penalty_scale": 1.0, "drawdown_penalty_scale": 1.0,
            "quality_scale": 0.5, "position_mgmt_scale": 0.8,
            "learning_rate": 2e-4, "ent_coef_phase1": 0.08, "gamma": 0.997,
            "phase1_end": 0.35,
        },
        {  # High fee penalty, high quality reward
            "fee_scale": 1.5, "opportunity_cost_scale": 0.8,
            "opportunity_cost_cap": 0.04, "opportunity_cost_threshold": 0.002,
            "churn_penalty_scale": 1.5, "drawdown_penalty_scale": 1.2,
            "quality_scale": 1.8, "position_mgmt_scale": 1.5,
            "learning_rate": 1.5e-4, "ent_coef_phase1": 0.12, "gamma": 0.993,
            "phase1_end": 0.3,
        },
        {  # Minimal penalties, max exploration
            "fee_scale": 0.1, "opportunity_cost_scale": 0.1,
            "opportunity_cost_cap": 0.01, "opportunity_cost_threshold": 0.005,
            "churn_penalty_scale": 0.1, "drawdown_penalty_scale": 0.1,
            "quality_scale": 0.1, "position_mgmt_scale": 0.1,
            "learning_rate": 5e-4, "ent_coef_phase1": 0.15, "gamma": 0.990,
            "phase1_end": 0.5,
        },
    ]
    for seed_params in SEED_TRIALS:
        study.enqueue_trial(seed_params)
    print(f"Enqueued {len(SEED_TRIALS)} seed trials")

completed = len([t for t in study.trials
                 if t.state in (optuna.trial.TrialState.COMPLETE,
                                optuna.trial.TrialState.PRUNED)])
remaining = max(0, N_SCREENING_TRIALS - completed)
print(f"Completed trials: {completed}, remaining: {remaining}")

if remaining > 0:
    study.optimize(objective, n_trials=remaining, timeout=None)
else:
    print(f"All {N_SCREENING_TRIALS} screening trials already completed.")

print(f"\nBest trial: #{study.best_trial.number}, score={study.best_trial.value:.4f}")

## Cell 5 — Screening Results & Analysis

In [ ]:
import json

results = []
for trial in study.trials:
    if trial.value is not None and trial.value > -999:
        row = {"trial": trial.number, "score": trial.value, "state": trial.state.name}
        row.update(trial.params)
        row.update({k: v for k, v in trial.user_attrs.items() if isinstance(v, (int, float, str))})
        results.append(row)

results_df = pd.DataFrame(results).sort_values("score", ascending=False)
print(f"Viable trials: {len(results_df)} / {len(study.trials)}")
print()

display_cols = [
    "trial", "score", "pnl_pct", "trades_per_ep", "win_rate",
    "direction_balance", "action_entropy", "long_trades", "short_trades",
    "ent_coef_phase1", "phase1_end",
]
available = [c for c in display_cols if c in results_df.columns]
print(results_df[available].head(15).to_string(index=False))

results_df.to_csv(RESULTS_CSV, index=False)
print(f"\nResults saved to {RESULTS_CSV}")

try:
    from optuna.visualization import (
        plot_optimization_history,
        plot_param_importances,
        plot_slice,
    )

    fig1 = plot_optimization_history(study)
    fig1.write_image(str(PLOTS_DIR / "screening_optimization_history.png"))
    fig1.show()

    fig2 = plot_param_importances(study)
    fig2.write_image(str(PLOTS_DIR / "screening_param_importances.png"))
    fig2.show()

    fig3 = plot_slice(study, params=["ent_coef_phase1", "phase1_end",
                                     "opportunity_cost_scale", "fee_scale"])
    fig3.write_image(str(PLOTS_DIR / "screening_slice_key_params.png"))
    fig3.show()

    print(f"Plots saved to {PLOTS_DIR}")
except Exception as e:
    print(f"Visualization error: {e}")

## Cell 6 — Phase 2: Validation (500k × top 10)

In [ ]:
print("=" * 60)
print("PHASE 2: Validation — 500k steps x top 10")
print("=" * 60)

top_trials = sorted(
    [t for t in study.trials if t.value is not None and t.value > -999],
    key=lambda t: t.value,
    reverse=True,
)[:10]

print(f"Validating {len(top_trials)} configs at {VALIDATION_TIMESTEPS:,} timesteps\n")

VALIDATION_CSV = DRIVE_DIR / "validation_results.csv"
validation_results = []

existing_validated = set()
if VALIDATION_CSV.exists():
    existing_df = pd.read_csv(VALIDATION_CSV)
    existing_validated = set(existing_df["screening_trial"].tolist())
    validation_results = existing_df.to_dict("records")

for i, trial_data in enumerate(top_trials):
    if trial_data.number in existing_validated:
        print(f"  [{i+1}/{len(top_trials)}] Trial #{trial_data.number} — already validated")
        continue

    print(f"  [{i+1}/{len(top_trials)}] Trial #{trial_data.number} "
          f"(screening={trial_data.value:.4f})")
    params = trial_data.params

    train_env = None
    try:
        base_reward_config = {
            k: params[k] for k in [
                "fee_scale", "opportunity_cost_scale", "opportunity_cost_cap",
                "opportunity_cost_threshold", "churn_penalty_scale",
                "drawdown_penalty_scale", "quality_scale", "position_mgmt_scale",
            ]
        }

        config = TrainingConfig(
            total_timesteps=VALIDATION_TIMESTEPS,
            learning_rate=params["learning_rate"],
            gamma=params["gamma"],
            ent_coef=params["ent_coef_phase1"],
            simple_actions=True,
            fixed_size_pct=0.10,
            fixed_leverage=10,
            phase1_end=params["phase1_end"],
            phase2_end=params["phase1_end"] + 0.3,
            eval_freq=VALIDATION_TIMESTEPS + 1,
        )

        envs = SubprocVecEnv([
            make_env(train_df, config, rank=r)
            for r in range(N_ENVS)
        ])
        train_env = VecNormalize(envs, norm_obs=False, norm_reward=True, clip_reward=10.0)

        eval_raw = TradingEnv(
            df=val_df,
            window_size=config.window_size,
            episode_length=config.episode_length,
            simple_actions=True,
            fixed_size_pct=0.10,
            fixed_leverage=10,
        )

        agent = create_agent(train_env, config, verbose=0)

        curriculum_cb = CurriculumCallback(
            config,
            base_reward_config=base_reward_config,
            phase3_fee_multiplier=1.0,
            phase3_churn_multiplier=1.0,
            ent_coef_phase1=params["ent_coef_phase1"],
            verbose=0,
        )

        agent.learn(
            total_timesteps=config.total_timesteps,
            callback=CallbackList([curriculum_cb]),
            progress_bar=False,
        )

        score, details = evaluate_agent(agent, eval_raw, n_episodes=EVAL_EPISODES)
        result = {
            "screening_trial": trial_data.number,
            "screening_score": trial_data.value,
            "validation_score": score,
            **{k: v for k, v in details.items() if isinstance(v, (int, float, str))},
            **params,
        }
        validation_results.append(result)
        pd.DataFrame(validation_results).to_csv(VALIDATION_CSV, index=False)

        print(f"    -> score={score:.4f}, pnl={details.get('pnl_pct', '?')}%, "
              f"dir_balance={details.get('direction_balance', '?')}, "
              f"L={details.get('long_trades', '?')}/S={details.get('short_trades', '?')}")

    except Exception as e:
        print(f"    -> FAILED: {e}")
        validation_results.append({
            "screening_trial": trial_data.number,
            "screening_score": trial_data.value,
            "validation_score": -1000.0,
            "error": str(e),
        })
        pd.DataFrame(validation_results).to_csv(VALIDATION_CSV, index=False)
    finally:
        if train_env is not None:
            try:
                train_env.close()
            except Exception:
                pass

## Cell 7 — Final Results & Best Parameters

In [ ]:
import json

val_df_results = pd.DataFrame(validation_results)
val_df_results = val_df_results.sort_values("validation_score", ascending=False)

print("=" * 60)
print("FINAL RESULTS — Ranked by 500k Validation Score")
print("=" * 60)
print()

display_cols = [
    "screening_trial", "screening_score", "validation_score",
    "pnl_pct", "trades_per_ep", "win_rate",
    "direction_balance", "long_trades", "short_trades",
]
available = [c for c in display_cols if c in val_df_results.columns]
print(val_df_results[available].to_string(index=False))

best_row = val_df_results.iloc[0]
best_trial_num = int(best_row["screening_trial"])
best_trial = next(t for t in study.trials if t.number == best_trial_num)

best_params = {
    "source": "optuna_simple_v3",
    "screening_trial": best_trial_num,
    "screening_score": float(best_row["screening_score"]),
    "validation_score": float(best_row["validation_score"]),
    "params": best_trial.params,
    "base_reward_config": {
        k: best_trial.params[k]
        for k in [
            "fee_scale", "opportunity_cost_scale", "opportunity_cost_cap",
            "opportunity_cost_threshold", "churn_penalty_scale",
            "drawdown_penalty_scale", "quality_scale", "position_mgmt_scale",
        ]
    },
    "curriculum": {
        "phase3_fee_multiplier": 1.0,
        "phase3_churn_multiplier": 1.0,
        "ent_coef_phase1": best_trial.params["ent_coef_phase1"],
        "phase1_end": best_trial.params["phase1_end"],
    },
    "ppo": {
        "learning_rate": best_trial.params["learning_rate"],
        "gamma": best_trial.params["gamma"],
    },
}

with open(BEST_PARAMS_JSON, "w") as f:
    json.dump(best_params, f, indent=2)

print(f"\nBest params saved to {BEST_PARAMS_JSON}")
print(f"\nBest configuration (trial #{best_trial_num}):")
print(json.dumps(best_params["params"], indent=2))

## Cell 8 — How to Apply Best Parameters

Use these parameters in `colab_train_simple.ipynb` for a full 2M-step training run.

In [ ]:
import json

with open(BEST_PARAMS_JSON) as f:
    bp = json.load(f)

p1_end = bp["curriculum"]["phase1_end"]
p2_end = p1_end + 0.3

print("=" * 60)
print("APPLY IN colab_train_simple.ipynb (Cell 3)")
print("=" * 60)
print()
print(f"config = TrainingConfig(")
print(f"    total_timesteps=2_000_000,")
print(f"    learning_rate={bp['ppo']['learning_rate']},")
print(f"    gamma={bp['ppo']['gamma']},")
print(f"    ent_coef={bp['curriculum']['ent_coef_phase1']},")
print(f"    simple_actions=True,")
print(f"    fixed_size_pct=0.10,")
print(f"    fixed_leverage=10,")
print(f"    phase1_end={p1_end},")
print(f"    phase2_end={p2_end},")
print(f")")
print()
print(f"base_reward_config = {json.dumps(bp['base_reward_config'], indent=4)}")
print()
print("# Pass to train():")
print(f"agent = train(")
print(f"    agent=agent, config=config, eval_env=eval_env,")
print(f"    base_reward_config=base_reward_config,")
print(f")")
print()
print(f"# Or directly to CurriculumCallback:")
print(f"curriculum_cb = CurriculumCallback(")
print(f"    config,")
print(f"    base_reward_config=base_reward_config,")
print(f"    phase3_fee_multiplier=1.0,")
print(f"    phase3_churn_multiplier=1.0,")
print(f"    ent_coef_phase1={bp['curriculum']['ent_coef_phase1']},")
print(f")")